# ML-10 - Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**What this is.** Not a report about a model - the instruction sheet an editor uses on Monday
morning. It answers one question: *of the pages I could open this week, which ones, in what order,
and why that one?*

**Three choices made before any code ran, because they change what the numbers mean:**

| Choice | What I did | Why |
|---|---|---|
| How the queue is scored | **Out-of-fold** - every page scored by a model that never saw its client | The queue carries a precision I measured, not one borrowed from a different experiment |
| What an archetype is | Derived from the **reason codes** | ML-08's top 50 came only from `page_1` and `top_3`, so a position-tier mapping would ship with three empty rows |
| How reason codes are made | **Rule-derived** from the page's own February numbers | An editor can check every one by looking at the page. They describe the page, not the model's internals - and this notebook says so rather than implying otherwise |

**What this playbook is decision-support for, and what it is not.** It ranks pages that were still
visible in search during the outcome month, so a person can decide what to open first. It does not
say a page is healthy, it does not say reviewing a page will produce clicks, and it is not a
production system.


## Setup - the same model that was validated

*Identical warehouse pull, eligibility funnel, features and model as ML-08 and ML-09. A playbook
that describes a different model than the one I validated would be worthless.*


In [22]:
# ---- Setup: identical to ML-08 / ML-09 ----
import duckdb, os, json, numpy as np, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # Colab Secret. Never pasted in a cell.
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")
con.sql("SET preserve_insertion_order=false;")

SEED = 42

BASE        = "hf://datasets/FlyRank/internship-warehouse"
FEAT_MONTHS = ["2026-01", "2026-02"]   # everything knowable BEFORE the decision
LABEL_MONTH = "2026-03"                # the outcome window
SEALED      = "2026-06"                # final panel month - never touched, saved for the capstone

FLOOR_FEB   = 500
FLOOR_MAR   = 100

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

def month_agg(m, need_pos):
    """One month partition -> one row per page. Cached, because a single 30M-row
    scan is long enough that a dropped Colab connection kills the whole run."""
    cache = f"work/outputs/_agg_{m}.parquet"
    if os.path.exists(cache):
        d = pd.read_parquet(cache); print(f"  {m}: {len(d)} pages (from cache)"); return d
    extra = (", SUM(gsc_sum_position) AS sumpos,"
             " COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days") if need_pos else ""
    f = f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet"
    d = con.sql(f"""
        SELECT content_hash_id,
               ANY_VALUE(client_hash_id) AS client_hash_id,
               SUM(gsc_impressions)      AS impr,
               SUM(gsc_clicks)           AS clicks{extra}
        FROM read_parquet('{f}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df()
    d.to_parquet(cache, index=False); print(f"  {m}: {len(d)} pages")
    return d

print("Reading 3 monthly partitions one at a time. Sealed month", SEALED, "is NOT touched.")
jan = month_agg("2026-01", False)
feb = month_agg("2026-02", True)
mar = month_agg("2026-03", False)

jan = jan.rename(columns={"impr": "impr_jan", "clicks": "clicks_jan"}).drop(columns=["client_hash_id"])
mar = mar.rename(columns={"impr": "impr_mar", "clicks": "clicks_mar"}).drop(columns=["client_hash_id"])
feb = feb.rename(columns={"impr": "impr_feb", "clicks": "clicks_feb",
                          "sumpos": "sumpos_feb", "active_days": "active_days_feb"})

p = feb.merge(jan, on="content_hash_id", how="outer").merge(mar, on="content_hash_id", how="outer")
dimc = con.sql(f"""SELECT content_hash_id, content_type, main_intent, word_count
                   FROM read_parquet('{BASE}/dim_content.parquet')""").df()
raw = p.merge(dimc, on="content_hash_id", how="inner")
print("Pages seen in at least one of the three months:", len(raw))


Reading 3 monthly partitions one at a time. Sealed month 2026-06 is NOT touched.
  2026-01: 121544 pages (from cache)
  2026-02: 153559 pages (from cache)
  2026-03: 176738 pages (from cache)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages seen in at least one of the three months: 203074


In [23]:
# ---- The ML-08 frame and machinery, unchanged ----
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

FEATS_NUM = ["impr_jan","clicks_jan","impr_feb","clicks_feb","ctr_jan","ctr_feb",
             "pos_feb","momentum","active_days_feb","word_count","noise"]
FEATS_CAT = ["content_type","main_intent"]
FEATS_ALL = FEATS_NUM + FEATS_CAT
MAX_ITER  = 200          # the value ML-08's inner grouped search chose. Not re-tuned.
N_SPLITS  = 5

def tier(p):
    if p <= 3:  return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

def build_frame():
    d = raw.copy()
    d = d[d["impr_feb"].notna() & d["impr_jan"].notna() & d["impr_mar"].notna()]
    d = d[d["impr_feb"] >= FLOOR_FEB]
    d = d[d["impr_mar"] > 0]
    d = d[d["impr_mar"] >= FLOOR_MAR]
    d = d[d["sumpos_feb"] > 0]
    d = d.sort_values("content_hash_id").reset_index(drop=True)
    rng = np.random.default_rng(SEED)
    d["ctr_feb"]  = d["clicks_feb"] * 100.0 / d["impr_feb"]
    d["ctr_jan"]  = d["clicks_jan"] * 100.0 / d["impr_jan"].replace(0, np.nan)
    d["pos_feb"]  = d["sumpos_feb"] / d["impr_feb"]
    d["momentum"] = d["impr_feb"] / d["impr_jan"].replace(0, np.nan)
    d["ctr_mar"]  = d["clicks_mar"] * 100.0 / d["impr_mar"]      # OUTCOME - never a feature
    d["noise"]    = rng.normal(size=len(d))
    d["tier_feb"] = d["pos_feb"].apply(tier)
    d = d.reset_index(drop=True)
    d[FEATS_NUM] = d[FEATS_NUM].astype("float64")
    d[FEATS_CAT] = d[FEATS_CAT].astype("object")
    return d

df     = build_frame()
groups = df["client_hash_id"].values
folds  = list(GroupKFold(n_splits=N_SPLITS).split(df, groups=groups))

def fold_label(frame, train_idx, test_idx):
    """25th-percentile CTR cut per February tier, FITTED ON TRAIN ONLY."""
    tr = frame.iloc[train_idx]
    cuts = tr.groupby("tier_feb")["ctr_mar"].quantile(0.25)
    glob = tr["ctr_mar"].quantile(0.25)
    y_tr = (tr["ctr_mar"] < tr["tier_feb"].map(cuts).fillna(glob)).astype(int).values
    te = frame.iloc[test_idx]
    y_te = (te["ctr_mar"] < te["tier_feb"].map(cuts).fillna(glob)).astype(int).values
    return y_tr, y_te

def fold_rule_scores(frame, train_idx, test_idx):
    """The frozen Week-4 rule, tier medians fitted on TRAIN only."""
    tr = frame.iloc[train_idx]
    med  = tr.groupby("tier_feb")["ctr_feb"].median()
    glob = tr["ctr_feb"].median()
    te = frame.iloc[test_idx]
    shortfall = (te["tier_feb"].map(med).fillna(glob) - te["ctr_feb"]).clip(lower=0)
    return (shortfall * te["impr_feb"]).values

def make_histgb(feats):
    cats = [c for c in feats if c in FEATS_CAT]
    nums = [c for c in feats if c not in FEATS_CAT]
    cat_tf = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="__NA__")),
                       ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    pre  = ColumnTransformer([("c", cat_tf, cats), ("n", "passthrough", nums)])
    mask = [True] * len(cats) + [False] * len(nums)
    return Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(
        categorical_features=mask, early_stopping=False, max_iter=MAX_ITER,
        learning_rate=0.06, max_leaf_nodes=31, l2_regularization=1.0, random_state=SEED))])

print(f"Frame: {len(df)} pages, {df['client_hash_id'].nunique()} clients, {N_SPLITS} client-grouped folds.")


Frame: 40152 pages, 24 clients, 5 client-grouped folds.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**How every page in this queue was scored.** Each page's score comes from a model trained on the
*other* clients - out-of-fold, exactly the arrangement ML-08 and ML-09 measured. Nothing here was
scored by a model that had already seen that client's pages. That costs a little accuracy against
training on the full panel, and buys the thing that matters more: the precision figure attached to
this queue is one I measured, not one carried over from a different experiment.

**Order.** Score descending, then February impressions descending, then page id - the same tie
policy used in every comparison since Week 4. Ties are a known failure of my Week-4 rule (its top
100 all shared one score), so the tie-break is written down rather than left to chance.

**Reason codes describe the page, not the model.** Each code is a fact an editor can check by
opening the page: this one gets a lot of impressions, this one converts below others at its search
position, this one is slipping month on month. They are **not** an account of what the model
internally weighted - a rule-derived label and a model explanation are different things and I am
not going to blur them. What they are for is disagreement: if a code contradicts what the editor
knows about that page, that is a signal to overrule the queue, and overruling it is a correct use.

**Archetypes group pages by why they were flagged**, and each archetype gets one action. A page
matches the first archetype it qualifies for, so every page has exactly one - a queue where a page
appears under three headings is not a queue.

**One archetype exists because the data contradicted me - and then partly un-contradicted me.**
I expected declining pages to be the priority: the industry "content decay, so refresh it" story.
Section 3 measures it. Across the whole panel the association is flat to slightly reversed - every
momentum bucket sits between 0.198 and 0.242 against a base rate of 0.225, with the *rising* bucket
highest. So momentum on its own barely separates anything, in either direction, and the decay story
is not carrying the weight it is usually given.

But the archetype table below tells a second story, and the two must be read together. Once you
condition on a page **already converting below its tier**, the slipping ones do under-capture more
often than the growing ones. Different populations, not a contradiction - and the honest conclusion
is that *below-tier conversion is what separates pages; month-on-month momentum adds very little on
top of it*. `growing_diluted` earns its place because dilution is a distinct editorial problem
needing a distinct action, not because growth predicts under-capture. `slipping_page` keeps its
action and loses the claim that decline is a priority signal on its own.


In [24]:
# ---- Out-of-fold scores: every page judged by a model that never saw its client ----
oof_score = np.full(len(df), np.nan)
oof_y     = np.full(len(df), -1, dtype=int)
rule_oof  = np.full(len(df), np.nan)
perfold_p50 = []          # ML-08's quantity, kept so the two are never confused

for tr_i, te_i in folds:
    y_tr, y_te = fold_label(df, tr_i, te_i)
    pipe = make_histgb(FEATS_ALL).fit(df.iloc[tr_i][FEATS_ALL], y_tr)
    s = pipe.predict_proba(df.iloc[te_i][FEATS_ALL])[:, 1]
    oof_score[te_i] = s
    oof_y[te_i]     = y_te
    rule_oof[te_i]  = fold_rule_scores(df, tr_i, te_i)
    # Top 50 WITHIN this fold's test set - the same measurement ML-08 reported.
    t = pd.DataFrame({"score": s,
                      "impr_feb": df.iloc[te_i]["impr_feb"].values,
                      "cid": df.iloc[te_i]["content_hash_id"].astype(str).values,
                      "y": y_te}).sort_values(["score", "impr_feb", "cid"],
                                              ascending=[False, False, True])
    perfold_p50.append(float(t["y"].head(50).mean()))

q = df.copy()
q["score"]     = oof_score
q["opportunity"] = oof_y          # the outcome, for measuring - NOT shown to an editor in advance
q["rule_score"] = rule_oof
assert not np.isnan(oof_score).any(), "a page was never scored out of fold"
print(f"Scored {len(q)} pages out of fold. Base rate: {q['opportunity'].mean():.3f}")

# ---- Reason codes: facts about the page, each one checkable by opening it ----
tier_med = q.groupby("tier_feb")["ctr_feb"].median()
hi_vol   = q["impr_feb"].quantile(0.75)
thin_cut = q["word_count"].quantile(0.25)

q["below_tier"]   = q["ctr_feb"] < q["tier_feb"].map(tier_med)
q["high_volume"]  = q["impr_feb"] >= hi_vol
q["zero_click"]   = q["clicks_feb"] == 0
q["slipping"]     = q["momentum"] < 0.90
q["rising"]       = q["momentum"] > 1.10
q["thin"]         = q["word_count"] < thin_cut
q["intermittent"] = q["active_days_feb"] < 20

CODE_TEXT = {
    "below_tier":   "converts below the typical page at its search position",
    "high_volume":  "high impressions - a small CTR gain is worth a lot here",
    "zero_click":   "no clicks at all in February despite real impressions",
    "slipping":     "impressions fell more than 10% from January",
    "rising":       "impressions rose more than 10% from January",
    "thin":         "short page relative to the rest of the portfolio",
    "intermittent": "appeared in search on fewer than 20 days in February",
}
CODES = list(CODE_TEXT)

def codes_for(row):
    return [c for c in CODES if bool(row[c])]

# ---- Archetypes: first match wins, so every page has exactly one ----
def archetype(r):
    if r["zero_click"]:                        return "zero_click_verify"
    if r["high_volume"] and r["below_tier"]:   return "big_under_converter"
    if r["momentum"] > 1.5 and r["below_tier"]: return "growing_diluted"
    if r["slipping"]    and r["below_tier"]:   return "slipping_page"
    if r["thin"]        and r["below_tier"]:   return "thin_under_converter"
    if r["below_tier"]:                        return "other_under_converter"
    return "watch_only"

q["archetype"] = q.apply(archetype, axis=1)

ACTIONS = pd.DataFrame([
    ("zero_click_verify",     "VERIFY FIRST - confirm the page is reachable, indexed and not a redirect "
                              "before touching copy. Only then treat as an under-converter.", "Editor + tech check", "20 min"),
    ("big_under_converter",   "Rewrite title and meta description. Highest value per hour - the traffic "
                              "already exists, the click does not.",                          "Editor",             "45 min"),
    ("growing_diluted",       "Check WHICH queries it now matches. Impressions grew more than 50% while "
                              "CTR sat below its tier - usually shown wider, matching worse. Tighten the "
                              "page to its actual intent rather than broadening it further.", "Editor",             "60 min"),
    ("slipping_page",         "Refresh: update facts, dates and examples, then request reindexing. "
                              "NOTE: across the whole panel, momentum barely separates - falling pages are "
                              "no more likely to under-capture than flat ones. Among already below-tier "
                              "pages it does carry signal. Read both tables in section 3 before treating "
                              "decline as a priority on its own.",                            "Editor",             "90 min"),
    ("thin_under_converter",  "Expand the page to answer the question fully, then revisit the title.",
                                                                                              "Editor + writer",   "3 h"),
    ("other_under_converter", "Read the page against the query intent. Decide rewrite vs refresh.",
                                                                                              "Editor",             "30 min"),
    ("watch_only",            "No action. Present for completeness, not recommended for review.",
                                                                                              "-",                  "-"),
], columns=["archetype", "recommended action", "who", "rough effort"])

print()
print("=" * 100)
print("ARCHETYPE -> ACTION")
print("=" * 100)
print(ACTIONS.to_string(index=False))

# How many pages actually fall into each archetype across the WHOLE queue - not just
# the top 50. Printed because the self-check claims no archetype is empty, and a claim
# in a self-check that nothing measured is exactly the kind of thing ML-09 was about.
print()
print("HOW MANY PAGES ARE IN EACH ARCHETYPE, ACROSS ALL", f"{len(q):,}", "PAGES")
spread = (q.groupby("archetype")
            .agg(pages=("score", "size"), under_captured=("opportunity", "mean"))
            .reindex(ACTIONS["archetype"]))
spread["under_captured"] = spread["under_captured"].round(3)
spread["share"] = (spread["pages"] / len(q)).map(lambda v: f"{v:.1%}")
print(spread.to_string())
empty = spread.index[spread["pages"].fillna(0) == 0].tolist()
print(f"\n  Archetypes with no members: {empty if empty else 'none'}")

# ---- The queue, ordered by the same tie policy used since Week 4 ----
q = q.sort_values(["score", "impr_feb", "content_hash_id"],
                  ascending=[False, False, True]).reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)
q["reason_codes"] = q.apply(lambda r: ";".join(codes_for(r)), axis=1)
q["why"] = q.apply(lambda r: "; ".join(CODE_TEXT[c] for c in codes_for(r)) or "no flag raised", axis=1)

print()
print("=" * 100)
print("TOP 15 OF THE QUEUE - what an editor sees")
print("=" * 100)
show = q.head(15)[["rank", "archetype", "tier_feb", "impr_feb", "ctr_feb", "score", "why"]].copy()
show["impr_feb"] = show["impr_feb"].map(lambda v: f"{int(v):,}")
show["ctr_feb"]  = show["ctr_feb"].map(lambda v: f"{v:.3f}%")
show["score"]    = show["score"].map(lambda v: f"{v:.3f}")
show["why"]      = show["why"].str.slice(0, 62)
print(show.to_string(index=False))
print()
print("The page id and client id exist in the exported file but are not printed here -")
print("pseudonymised hashes are still identifiers and this notebook is public.")


Scored 40152 pages out of fold. Base rate: 0.225

ARCHETYPE -> ACTION
            archetype                                                                                                                                                                                                                                                                                                                   recommended action                 who rough effort
    zero_click_verify                                                                                                                                                                                                VERIFY FIRST - confirm the page is reachable, indexed and not a redirect before touching copy. Only then treat as an under-converter. Editor + tech check       20 min
  big_under_converter                                                                                                                                                     

## 2. Intended use and limits

*Who uses this, for what - and where it stops being valid.*

**Who it is for.** Molnar's interpretability chapter borrows a role model from Tomsett et al., and
it is more useful here than "this is for editors", because each role needs a different thing:

| Role | Who, here | What they get from this playbook |
|---|---|---|
| Creator | me | The record of how the queue was built and validated |
| Operator | whoever re-runs this notebook | The exports, and the monitoring triggers in section 4 |
| Executor | the editor choosing what to open | The rank, the action, and the reason codes to argue with |
| Decision subject | the pages, and the clients whose pages are or are not reviewed | The scope statement below - what being absent from the queue does and does not mean |
| Auditor | FlyRank reviewers, and readers of the paper | Every number here traces to a committed metrics file |
| Data subject | the clients whose search data trained it | Pseudonymised throughout; no page, client or query is named |

**What it is for.** Ordering a week of editorial attention. An editor can review roughly 50 pages a
week against a portfolio in the tens of thousands, so the scarce thing is not analysis, it is
attention - and the only question worth answering is *which one first*.

**Where it stops being valid.** These are measured limits, not disclaimers:

- **Scope.** The queue covers pages that were **still receiving search impressions during the
  outcome month**. A page that vanished from search entirely cannot be scored at all - March CTR is
  a rate, and a rate needs a denominator. Those pages are structurally absent, and they are arguably
  the ones most in need of attention. Measured in ML-09: 979 of 41,234 February-eligible pages.
- **Roughly half the signal is borrowed from click history.** Remove the four click-derived
  features and precision@50 falls from 0.784 to 0.604 - still ahead of the rule, by a much narrower
  margin. Measured once, in ML-08.
- **A two-month feature window cannot tell "this page under-converts" from "February was a bad
  month for this page."** Every one of the five worst-fold misses in ML-09 had a February CTR of
  exactly 0.000% and recovered in March on its own. This is a property of the window, not a bug.
- **Five folds, and three of them hold a single test client.** The fold-to-fold spread is wide
  (0.60 to 0.90). Treat the headline as directional.
- **Nothing here is causal.** The model observed which pages under-captured clicks. It says nothing
  about whether editing them produces clicks. That would need a controlled comparison nobody has run.


In [25]:
# ---- Cost and value: the queue length IS the decision threshold ----
def precision_at(k):
    return float(q["opportunity"].head(k).mean())

base = float(q["opportunity"].mean())
rows = []
for k in [10, 20, 30, 50, 100, 200]:
    p = precision_at(k)
    rows.append({"queue length K": k, "precision@K": round(p, 3),
                 "x base rate": round(p / base, 2),
                 "real opportunities found": int(round(p * k)),
                 "pages reviewed for nothing": int(round((1 - p) * k))})
costval = pd.DataFrame(rows)

print("=" * 92)
print(f"COST AND VALUE - base rate {base:.3f}. An editor reviews about 50 pages a week.")
print("=" * 92)
print(costval.to_string(index=False))
print()
print("There is no probability threshold to tune here - K is the threshold. A shorter queue is a")
print("more accurate queue; a longer one finds more real opportunities and wastes more hours.")
print("The honest framing for a decision-maker is the last two columns, not the precision.")
print()
print("  Read the small-K end carefully. At K=10 a single wrong page costs 10 percentage")
print("  points, so that number is noisy in a way the K=100 number is not. If precision at")
print("  K=10 comes out BELOW precision at K=20 or K=30, that is one miss near the top, not")
print("  a pattern - and it is not evidence that a longer queue is more accurate.")

print()
print("=" * 92)
print("TWO DIFFERENT PRECISION@50 FIGURES. THEY ARE NOT INTERCHANGEABLE.")
print("=" * 92)
print(f"  This queue, pooled across all folds : {precision_at(50):.3f}")
print(f"  ML-08's measurement, per fold       : {np.mean(perfold_p50):.3f}"
      f"   (folds: {[round(v, 2) for v in perfold_p50]})")
print()
print("  Same model, same rows, same folds. The difference is WHICH 50 pages are counted.")
print(f"  Pooled takes the best 50 of {len(q):,}. Per-fold takes the best 50 of roughly")
print(f"  {len(q)//N_SPLITS:,}, five separate times. Creaming the top of a much larger pool is an")
print("  easier task, so the pooled number is higher - the model did not improve.")
print()
print("  Which to quote: the POOLED figure describes this queue, because this queue is")
print("  pooled. The PER-FOLD figure is the one that compares to the Week-4 rule and to")
print("  ML-08. Putting them side by side as if one beat the other would be a false claim.")

# ---- Where the queue's pages actually come from ----
print()
print("ARCHETYPE MIX IN THE TOP 50, AND HOW OFTEN EACH ONE WAS RIGHT")
mix = (q.head(50).groupby("archetype")
         .agg(pages=("rank", "size"), correct=("opportunity", "sum"))
         .assign(precision=lambda t: (t["correct"] / t["pages"]).round(2))
         .sort_values("pages", ascending=False))
print(mix.to_string())
print()
print("This table is the evidence behind section 3. An archetype that is right less often than")
print("the others is not one to trust less - it is one that needs a human check before acting.")

print()
print("SCOPE, IN NUMBERS")
print(f"  pages in the queue                    : {len(q):,}")
print(f"  clients represented                   : {q['client_hash_id'].nunique()}")
print(f"  feature months                        : {', '.join(FEAT_MONTHS)}")
print(f"  outcome month                         : {LABEL_MONTH}")
print(f"  month deliberately never queried      : {SEALED}")


COST AND VALUE - base rate 0.225. An editor reviews about 50 pages a week.
 queue length K  precision@K  x base rate  real opportunities found  pages reviewed for nothing
             10        0.900         3.99                         9                           1
             20        0.950         4.22                        19                           1
             30        0.967         4.29                        29                           1
             50        0.940         4.17                        47                           3
            100        0.830         3.68                        83                          17
            200        0.820         3.64                       164                          36

There is no probability threshold to tune here - K is the threshold. A shorter queue is a
more accurate queue; a longer one finds more real opportunities and wastes more hours.
The honest framing for a decision-maker is the last two columns, not the pr

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Molnar puts the purpose of an explanation plainly: *if the explanation conflicts with domain
knowledge, the decision maker might question the forecast and investigate.* The reason codes exist
so the editor can **overrule** the queue. Every rule below is written to make that easier.

### Check before acting

1. **Open the page.** Every reason code is a claim about the page that can be checked by looking at
   it. If the code says "short page" and the page is long, the queue is wrong about this row -
   report it rather than working around it.
2. **`zero_click_verify` gets a technical check first.** A page with real impressions and zero
   clicks in a month is more often a broken thing - a redirect, a noindex, a wrong canonical - than
   a copywriting problem. Rewriting the title of a page nobody can reach wastes the hour twice.
   This is also where the model is most often wrong: every one of ML-09's worst-fold misses had
   exactly this profile and recovered in March without anyone touching it.
3. **Check February against a longer history before calling a page a decay case.** The model sees
   two months. A page that had one quiet month looks identical to a page in genuine decline.
4. **Sanity-check the client mix.** If a single client fills the top of the queue, that is worth a
   look before it becomes a week of work on one account. Three of the five validation folds held a
   single client, so client concentration is a known weak spot.

### The no-go list

- **The queue is not a safe list.** It says which pages to review *first*. It never says a page is
  fine. Absence from the queue means low score, no score, or outside the population - and those are
  not the same thing. Acting on absence is the one misuse that would do real harm.
- **Do not auto-apply any recommended action.** No automatic title rewrites, no automatic
  reindexing requests. Every action is a suggestion for a person who will look at the page.
- **Do not retrain automatically on a drift alert.** Data-quality bugs look exactly like drift, and
  retraining on broken data bakes the bug into the model. A drift alert is a reason to investigate,
  never a trigger to retrain.
- **Do not present the score as a probability of anything real.** It orders pages. It is not "an
  84% chance this page will improve."
- **Do not claim the review caused the outcome.** If a reviewed page improves, this design cannot
  separate the edit from the page recovering on its own - which we have measured happening.
- **Do not use it on a client with no January or February history.** The features do not exist and
  the page will not be in the frame; a score for one would be fabricated.
- **Do not query the sealed month to check whether the queue was right.** June is spent once, on
  the capstone.


In [26]:
# ---- Evidence for the review rules, rather than assertion ----
print("=" * 92)
print("RULE 2 - IS zero_click_verify REALLY THE SHAKY ONE?")
print("=" * 92)
zc = q.groupby("zero_click").agg(
        pages=("rank", "size"),
        made_top50=("rank", lambda s: int((s <= 50).sum())),
        precision_overall=("opportunity", "mean"))
zc["precision_overall"] = zc["precision_overall"].round(3)
print(zc.to_string())
print("  (made_top50 counts pages of that kind inside the top 50. An earlier version of this")
print("   cell labelled it as a share and was ambiguous - counts are unambiguous.)")
top50 = q.head(50)
if top50["zero_click"].any():
    a = float(top50[top50["zero_click"]]["opportunity"].mean())
    b = float(top50[~top50["zero_click"]]["opportunity"].mean())
    print(f"\n  In the top 50: zero-click pages correct {a:.0%}, everything else {b:.0%}.")
    print(f"  zero-click pages are {top50['zero_click'].mean():.0%} of the top 50.")
else:
    print("\n  No zero-click pages reached the top 50 in this run.")

print()
print("=" * 92)
print("RULE 4 - HOW CONCENTRATED IS THE TOP OF THE QUEUE?")
print("=" * 92)
conc = (q.head(50).groupby("client_hash_id").size().sort_values(ascending=False))
print(f"  clients represented in the top 50 : {len(conc)} of {q['client_hash_id'].nunique()}")
print(f"  largest single client's share     : {conc.iloc[0] / 50:.0%}")
print("  (client ids not printed - pseudonymised hashes are still identifiers.)")

print()
print("=" * 92)
print("THE DECAY / REFRESH INSIGHT")
print("=" * 92)
decay = (q.groupby(pd.cut(q["momentum"], [0, .75, .9, 1.1, 1.5, np.inf],
                          labels=["falling >25%", "falling 10-25%", "flat", "rising 10-50%", "rising >50%"]),
                   observed=True)
           .agg(pages=("rank", "size"), under_captured=("opportunity", "mean"),
                median_feb_ctr=("ctr_feb", "median")))
decay["under_captured"]  = decay["under_captured"].round(3)
decay["median_feb_ctr"]  = decay["median_feb_ctr"].round(3)
print(decay.to_string())
print()
print("Read this as: does losing impressions month-on-month go with under-capturing clicks the")
print("month after? The 'falling' rows are the decay/refresh case the industry story predicts.")
print()
print("  In this panel the association runs the OTHER WAY. Pages gaining impressions")
print("  under-capture more often than pages losing them. The median-CTR column shows the")
print("  likely mechanism: CTR falls steadily as impressions rise, which is what happens when")
print("  a page is shown for a wider set of queries it matches less well.")
print("  This is a negative result for the refresh-the-declining-page recommendation, and it")
print("  is why the growing_diluted archetype exists and why slipping_page carries a caveat.")


RULE 2 - IS zero_click_verify REALLY THE SHAKY ONE?
            pages  made_top50  precision_overall
zero_click                                      
False       33731          44              0.178
True         6421           6              0.474
  (made_top50 counts pages of that kind inside the top 50. An earlier version of this
   cell labelled it as a share and was ambiguous - counts are unambiguous.)

  In the top 50: zero-click pages correct 83%, everything else 95%.
  zero-click pages are 12% of the top 50.

RULE 4 - HOW CONCENTRATED IS THE TOP OF THE QUEUE?
  clients represented in the top 50 : 8 of 24
  largest single client's share     : 44%
  (client ids not printed - pseudonymised hashes are still identifiers.)

THE DECAY / REFRESH INSIGHT
                pages  under_captured  median_feb_ctr
momentum                                             
falling >25%     4913           0.216           0.262
falling 10-25%   3892           0.198           0.260
flat             5783

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**The hard part first: we cannot measure whether the queue was right until the month closes.**
Whether a page under-captured clicks in March is only knowable in April. That is feedback delay, and
it means model quality cannot be watched directly. So the monitoring here is all **proxy** - watching
the inputs and the outputs for signs that the world changed, because the score itself will not tell
us.

**Three things can go wrong and they are different.**

| | What shifts | How we would see it | Why it matters here |
|---|---|---|---|
| **Data drift** | the input features, P(X) | compare this month's feature distributions to the reference | the model is being asked about pages unlike the ones it learned from |
| **Prediction drift** | our own scores, P(Y) | the score distribution moves without the inputs moving much | the best proxy we have, since real quality arrives a month late |
| **Concept drift** | the relationship, P(Y\|X) | **we cannot see this directly** | this is the real risk - if search changes how results are displayed, position and CTR stop relating the way they did, and the model is quietly wrong while every input looks normal |

**Checks run in this order, and the order matters.** Data quality first - completeness, no negative
impressions, no explosion in the `avg_position = 0` sentinel that means "no data" rather than rank
zero. Only then distribution checks. Reversed, every drift alert costs an investigation just to rule
out a pipeline bug.

**Which features to watch, and why not all of them.** ML-08 measured permutation importance against
an injected pure-noise column. Features that could not beat noise are not worth alerting on -
watching them only generates false alarms. The cell below names the ones worth watching, from that
measurement rather than from intuition.

**How big a shift is big enough.** Rather than adopting a threshold from a blog post, the cell below
measures the **January to February** shift on our own features - a month-to-month move in a period
when the model worked. Statistical tests are deliberately not used: on 40,152 rows they flag
differences far too small to act on, so the measure is PSI.

**Measuring that baseline changed the design, which is the reason for measuring it.** My first
version set the trigger at twice the largest normal move across all watched features. Page
impressions turn out to move enormously between ordinary months in a growing portfolio - far past
the conventional 0.25 "major shift" line - so that rule produced an alarm that could never fire. A
feature whose *normal* month is already a major shift cannot detect drift at all. So impressions are
now watched for **data quality** only, and the drift sentinel is click-through rate, which is
stable month to month. The cell below prints which features passed that test and which were
rejected, with the numbers.

**What to do when a trigger fires** - escalating, and only the first step is automatic:

1. **Investigate.** Is it a real shift, a data-quality bug, or a false alarm? Nothing else happens
   until this is answered.
2. **Retrain** - on a cadence, not on the alert. Monthly, when the new outcome month closes and real
   labels exist. Retraining on a drift alert means retraining on data nobody has checked.
3. **Intervene in the process** without retraining: shorten the queue, or stop issuing it for the
   affected client, and hand those pages to a person.
4. **Redesign** if the same trigger keeps firing - a wider feature window, or dropping a feature
   that keeps drifting.


In [27]:
# ---- What a normal month-to-month shift looks like, on our own features ----
WATCH = ["ctr_feb", "pos_feb", "impr_feb", "momentum", "active_days_feb"]

def psi(reference, current, bins=10):
    """Population Stability Index. Chosen over a hypothesis test because on 40k rows a
    test flags differences far too small to act on. Higher = further apart."""
    ref = pd.Series(reference).dropna(); cur = pd.Series(current).dropna()
    if len(ref) == 0 or len(cur) == 0:
        return float("nan")
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return float("nan")
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, bins=edges)[0] / len(ref)
    c = np.histogram(cur, bins=edges)[0] / len(cur)
    r = np.clip(r, 1e-6, None); c = np.clip(c, 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))

# January versions of the same quantities, on the same pages
ref_jan = pd.DataFrame({
    "ctr_feb":         q["ctr_jan"],
    "pos_feb":         np.nan,                 # no January position in the pull - stated, not faked
    "impr_feb":        q["impr_jan"],
    "momentum":        np.nan,
    "active_days_feb": np.nan,
})

print("=" * 92)
print("WHAT A NORMAL MONTH LOOKS LIKE - January vs February, same pages")
print("=" * 92)
base_rows = []
for f in WATCH:
    val = psi(ref_jan[f], q[f]) if ref_jan[f].notna().any() else float("nan")
    base_rows.append({"feature": f, "Jan->Feb PSI": (round(val, 3) if val == val else "not measurable"),
                      "watched?": "yes"})
baseline = pd.DataFrame(base_rows)
print(baseline.to_string(index=False))
print()
print("  Where a January equivalent does not exist in this pull, it says so instead of")
print("  inventing a number. Those features get a threshold only once a second month of")
print("  position data is pulled - noted as an open item rather than papered over.")

# ---- Which features can serve as drift sentinels at all ----
# A feature whose NORMAL month-to-month move is already past the conventional
# "major shift" line (PSI 0.25) cannot detect drift: any threshold loose enough
# to avoid firing every month is too loose to fire on anything.
MAJOR_SHIFT = 0.25
measurable = {r["feature"]: float(r["Jan->Feb PSI"])
              for r in base_rows if r["Jan->Feb PSI"] != "not measurable"}
usable   = {f: v for f, v in measurable.items() if v <= MAJOR_SHIFT}
too_noisy = {f: v for f, v in measurable.items() if v >  MAJOR_SHIFT}

print()
print("=" * 92)
print("WHICH FEATURES CAN ACTUALLY SERVE AS DRIFT SENTINELS")
print("=" * 92)
for f, v in usable.items():
    print(f"  USABLE     {f:<16} normal move PSI {v:.3f}  - stable enough that a shift means something")
for f, v in too_noisy.items():
    print(f"  REJECTED   {f:<16} normal move PSI {v:.3f}  - already past the {MAJOR_SHIFT} 'major shift' line")
    print(f"             in an ordinary month. A threshold above this would never fire.")
if too_noisy:
    print()
    print("  This is the point of measuring a baseline instead of copying a threshold from a")
    print("  blog post. My first version set the trigger at twice the largest normal move,")
    print("  which produced an alarm that could not go off. The measurement caught it.")

if usable:
    worst   = max(usable.values())
    trigger = round(max(0.10, worst * 2), 2)
else:
    worst, trigger = float("nan"), 0.20

print()
print("=" * 92)
print("THE TRIGGERS")
print("=" * 92)
print(f"  Sentinel features                          : {', '.join(usable) if usable else 'none usable'}")
print(f"  Largest normal move among them             : {worst:.3f}" if worst == worst
      else "  Largest normal move among them             : not measurable this run")
print(f"  Drift trigger set at                       : PSI > {trigger}")
print("    - twice the largest normal move among usable sentinels, floored at 0.10.")
print("      Investigate; do NOT auto-retrain.")
print("    - page impressions are watched for DATA QUALITY (see below) but are not a")
print("      drift sentinel: their ordinary month-to-month variation is far too large.")
print()
print("  Data-quality triggers, checked BEFORE any drift check:")
print("    - any negative impressions or clicks")
print(f"    - share of pages with avg_position = 0 (the 'no data' sentinel) rising above 2x")
print("      its current share")
print("    - a new value appearing in content_type or main_intent")
print("    - more than 5% of word_count missing where it was previously present")
print()
print("  Prediction-drift trigger:")
print(f"    - median queue score moves more than 0.10 from this run's {q['score'].median():.3f}")
print("    - or the archetype mix of the top 50 changes by more than half")
print()
print("  Retrain cadence: monthly, when a new outcome month closes and real labels exist.")
print("  Not on a drift alert - a drift alert is a reason to look, not a reason to retrain.")


WHAT A NORMAL MONTH LOOKS LIKE - January vs February, same pages
        feature   Jan->Feb PSI watched?
        ctr_feb          0.031      yes
        pos_feb not measurable      yes
       impr_feb          1.346      yes
       momentum not measurable      yes
active_days_feb not measurable      yes

  Where a January equivalent does not exist in this pull, it says so instead of
  inventing a number. Those features get a threshold only once a second month of
  position data is pulled - noted as an open item rather than papered over.

WHICH FEATURES CAN ACTUALLY SERVE AS DRIFT SENTINELS
  USABLE     ctr_feb          normal move PSI 0.031  - stable enough that a shift means something
  REJECTED   impr_feb         normal move PSI 1.346  - already past the 0.25 'major shift' line
             in an ordinary month. A threshold above this would never fire.

  This is the point of measuring a baseline instead of copying a threshold from a
  blog post. My first version set the trigger at t

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ - your paper builds on these
files.*

Three destinations, three different rules, and the repo's own `.gitignore` enforces them:

| File | Where | Committed? | Why |
|---|---|---|---|
| `w07_action_queue.csv` | `work/outputs/` | **No** - `work/**/*.csv` is blocked | It is a data export. The leak-guard blocks it by design and this notebook regenerates it |
| `ml10_precision_at_k.png`, `ml10_archetype_mix.png` | `work/figures/` | Yes | The paper reuses these directly |
| `w07_playbook_metrics.json` | `work/outputs/` | Yes | The receipts. Every number quoted in the paper traces back here |

The queue CSV carries the hashed page and client ids, because the paper needs a queue that can be
regenerated and checked - but nothing in the committed figures, JSON or notebook output names a page,
a client or a query.


In [28]:
# ---- Exports. Three destinations, three different rules. ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

QUEUE_N = 500
queue_cols = ["rank", "content_hash_id", "client_hash_id", "archetype", "reason_codes",
              "tier_feb", "impr_feb", "ctr_feb", "score"]
queue_out = q.head(QUEUE_N)[queue_cols].copy()
queue_path = "work/outputs/w07_action_queue.csv"
queue_out.to_csv(queue_path, index=False)
print(f"queue  -> {queue_path}  ({len(queue_out)} rows)   [NOT committed: work/**/*.csv is git-ignored]")

# Figure 1 - precision against queue length, with the base rate as the floor
fig, ax = plt.subplots(figsize=(7, 4.2))
ks = costval["queue length K"]
ax.plot(ks, costval["precision@K"], marker="o", color="#2F6F4E", label="precision@K")
ax.axhline(base, ls="--", color="#b23b3b", label=f"base rate {base:.3f}")
ax.set_xlabel("queue length K (pages handed to the editor)")
ax.set_ylabel("share that really under-captured clicks")
ax.set_title("A shorter queue is a more accurate queue")
ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig("work/figures/ml10_precision_at_k.png", dpi=150)
plt.close(fig)
print("figure -> work/figures/ml10_precision_at_k.png   [committed]")

# Figure 2 - what the top 50 is made of, and how often each archetype was right
fig, ax = plt.subplots(figsize=(7, 4.2))
m = mix.sort_values("pages")
ax.barh(m.index, m["pages"], color="#1F3864")
for i, (n, r) in enumerate(m.iterrows()):
    ax.text(r["pages"] + 0.4, i, f"{r['precision']:.0%} correct", va="center", fontsize=9, color="#5b6470")
ax.set_xlabel("pages in the top 50")
ax.set_title("What the top of the queue is made of")
ax.set_xlim(0, m["pages"].max() * 1.45)
fig.tight_layout()
fig.savefig("work/figures/ml10_archetype_mix.png", dpi=150)
plt.close(fig)
print("figure -> work/figures/ml10_archetype_mix.png    [committed]")

# Receipts
metrics = {
    "notebook": "ML-10 content action playbook",
    "scoring": "out-of-fold, GroupKFold on client_hash_id, n_splits=5, HistGB max_iter=200",
    "population": {"pages": int(len(q)), "clients": int(q["client_hash_id"].nunique()),
                   "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH,
                   "sealed_month_untouched": SEALED,
                   "scope_note": "pages still receiving search impressions in the outcome month"},
    "base_rate": round(base, 3),
    "precision_at_50_pooled": round(precision_at(50), 3),
    "precision_at_50_per_fold_mean": round(float(np.mean(perfold_p50)), 3),
    "precision_at_50_per_fold": [round(v, 3) for v in perfold_p50],
    "precision_note": ("pooled and per-fold are DIFFERENT quantities and must not be compared. "
                       "Pooled describes this queue. Per-fold is the figure that compares to "
                       "ML-08 and to the Week-4 rule."),
    "precision_at_k": {int(r["queue length K"]): float(r["precision@K"]) for _, r in costval.iterrows()},
    "lift_over_base_rate": {int(r["queue length K"]): float(r["x base rate"]) for _, r in costval.iterrows()},
    "archetype_mix_top50": {k: int(v) for k, v in mix["pages"].items()},
    "archetype_precision_top50": {k: float(v) for k, v in mix["precision"].items()},
    "monitoring": {"watched_features": WATCH,
                   "drift_sentinels": list(usable),
                   "rejected_as_sentinel": {f: round(v, 3) for f, v in too_noisy.items()},
                   "rejection_rule": f"normal Jan->Feb PSI above {MAJOR_SHIFT} means the feature "
                                     f"cannot detect drift; watched for data quality instead",
                   "psi_trigger": trigger,
                   "retrain_cadence": "monthly, on new labels - never on a drift alert"},
    "decay_finding": ("declining pages were NOT more likely to under-capture than flat pages in "
                      "this panel; pages growing >50% in impressions were the most likely. "
                      "Negative result for the refresh-the-declining-page recommendation."),
    "queue_export": {"path": queue_path, "rows": int(len(queue_out)), "committed": False,
                     "reason": "work/**/*.csv is blocked by the repo leak-guard; this notebook regenerates it"},
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("receipts -> work/outputs/w07_playbook_metrics.json   [committed]")
print()
print("Nothing in the committed figures, JSON or printed output names a page, a client or a query.")


queue  -> work/outputs/w07_action_queue.csv  (500 rows)   [NOT committed: work/**/*.csv is git-ignored]
figure -> work/figures/ml10_precision_at_k.png   [committed]
figure -> work/figures/ml10_archetype_mix.png    [committed]
receipts -> work/outputs/w07_playbook_metrics.json   [committed]

Nothing in the committed figures, JSON or printed output names a page, a client or a query.


## What the playbook says, in one page

**The most useful thing the queue does is set half the portfolio aside.** Of 40,152 eligible pages,
**20,061 - exactly 50% - are `watch_only`**, and their under-capture rate is **0.083** against a
panel base rate of **0.225**. An editor who ignores that half is ignoring pages that under-capture
less than a third as often as average. Everything else in this notebook is about ordering the other
half; this line is about not spending a week in the wrong half.

**The queue is accurate at the top, and the number that says so is not the one from ML-08.** Pooled
across all folds, precision@50 is **0.940** - 47 of 50 pages an editor opens really did under-capture
clicks, three did not. ML-08's per-fold figure for the same model on the same rows is **0.784**
(folds 0.90 / 0.78 / 0.84 / 0.80 / 0.60, reproduced exactly here, which is the evidence the model is
genuinely unchanged). These are different measurements: pooled takes the best 50 of 40,152, per-fold
takes the best 50 of roughly 8,030 five times over. The pooled figure describes this queue; the
per-fold figure is the one that compares to the Week-4 rule. Quoting the first as an improvement on
the second would be a false claim, and the notebook says so where both are printed.

**Queue length is the only dial, and the curve is flatter than it looks.** Precision runs 0.900 at
K=10, 0.950 at K=20, 0.967 at K=30, 0.940 at K=50, then 0.830 at K=100 and 0.820 at K=200. The dip
at K=10 is one wrong page, not a pattern - at that length a single miss costs ten points. The real
decision sits between K=50 and K=100: fifty pages yields 47 real opportunities and 3 wasted hours;
a hundred yields 83 and 17. Doubling the week's work roughly doubles the finds and multiplies the
waste by nearly six.

**What separates pages is below-tier conversion. Momentum adds very little.** I expected the
opposite - the content-decay story says find the declining pages. Across the whole panel every
momentum bucket sits between 0.198 and 0.242 around a 0.225 base rate, with the *rising* bucket
highest and the mildly-falling bucket lowest. Median February CTR falls steadily as impressions rise
(0.262% down to 0.152%), which is what dilution looks like: a page shown for a wider set of queries
it matches less well. Conditioned on already converting below tier, slipping pages do carry signal
(0.365 against 0.333 for growing ones) - so the two tables have to be read together, and the honest
summary is that decline is not a priority signal on its own.

**Zero-click pages are the strongest bulk signal and the least reliable at the very top.** Both are
true and neither cancels the other. Across the panel, 6,421 pages had real impressions and no clicks
in February, and **47.4%** of them under-captured in March against **17.8%** for everything else -
more than double. But inside the top 50 they are right 83% of the time against 95% for the rest.
That combination is exactly why they are ranked first for a *verification* step rather than for a
rewrite: they are worth looking at, and they are where the model is most likely to be wrong.

**The largest operational risk is not accuracy, it is concentration.** The top 50 draws from **8 of
24 clients, and a single client holds 44% of it**. Handed over without comment, that is a week in
which nearly half an editor's time goes to one account. Nothing in the model is wrong here - the
pages are genuine - but the person planning the week needs to see it before they start.

**The monitoring design changed because measuring the baseline broke my first version.** I set the
drift trigger at twice the largest normal month-to-month move. Page impressions shift with a PSI of
**1.346** between two ordinary months - five times past the conventional 0.25 "major shift" line - so
that rule produced an alarm at PSI > 2.69 that could never have fired. A feature whose normal month
is already a major shift cannot detect drift at all. Impressions are now a data-quality check;
click-through rate, stable at **0.031**, is the drift sentinel, with the trigger at **0.10**. Copying
0.2 from a blog post would have looked more professional and told me nothing.

**What I would do next, in order.** Pull a second month of position data so `pos_feb`, `momentum` and
`active_days_feb` get real drift baselines instead of "not measurable". Add a client cap to the queue
so no single account can take more than a stated share of a week. And widen the feature window past
two months, because the one failure mode that keeps recurring - a page that had a quiet month versus
a page that genuinely under-converts - is invisible to a two-month view and no amount of tuning will
fix it.

**What this playbook does not do.** It does not say a page is healthy, it cannot see pages that left
search entirely, and it says nothing about whether editing a page produces clicks. It orders a week
of attention, and that is the whole claim.


## Self-check

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere - only pseudonymised hash ids, and the
      committed outputs print none of them
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Ranked actions with reason codes an editor can check by opening the page
- [x] Archetype -> action mapping, one archetype per page, with the population of every
      archetype printed rather than asserted - including any that came out empty
- [x] The decay / refresh case measured rather than asserted - and the measurement came out
      against my expectation, so the recommendation was changed rather than the framing
- [x] Pooled and per-fold precision@50 both reported, with a plain statement that they are
      different quantities and must not be compared
- [x] Intended use written by stakeholder role, not as one audience
- [x] Limits are measured numbers carried from ML-08 and ML-09, not disclaimers
- [x] Human-review rules, each backed by evidence printed in the notebook
- [x] A no-go list that names what must never be automated
- [x] Cost/value framed as queue length, with pages-reviewed-for-nothing shown next to precision
- [x] Monitoring triggers calibrated against a real month-to-month shift, not a borrowed threshold
- [x] Retrain on a cadence; a drift alert investigates, never retrains
- [x] Queue exported for the paper; figures to `work/figures/`; receipts to a committed JSON
- [x] Committed to `work/notebooks/` - then submit the repo URL on the card
